### RAGAS

**RAGAS (Retrieval-Augmented Generation Assessment)** — библиотека с открытым исходным кодом для автоматической оценки качества RAG-пайплайнов.

* Поддерживает **локальные и облачные LLM** (включая mistral, llama3 через Ollama).
* **Не оценивает саму LLM**, а оценивает **работу RAG-пайплайна в целом.**

#### Шаг 1. Установка зависимостей

In [1]:
pip install ragas==0.4.3 langchain==0.2.16 langchain-core==0.2.41 langchain-community==0.2.17 langchain-openai==0.1.25 datasets sentence-transformers pydantic openai -q

Note: you may need to restart the kernel to use updated packages.


#### Шаг 2. Подготовка данных

В RAGAS 0.4+ используется **специальный формат данных:**

In [2]:
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset

dataset = EvaluationDataset(samples=[
    SingleTurnSample(
        user_input="Какой инструмент используется для миграций базы данных?",
        response="Для миграций базы данных используется Alembic.",
        retrieved_contexts=["Alembic используется для управления миграциями базы данных в SQLAlchemy."]
    ),
    SingleTurnSample(
        user_input="Что такое FastAPI?",
        response="FastAPI — это современный Python-фреймворк для создания API.",
        retrieved_contexts=["FastAPI — это современный Python-фреймворк для создания API."]
    )
])

/opt/anaconda3/envs/LLM_course/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Шаг 3. Настройка LLM и embedding-модели

In [3]:
import asyncio
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings import HuggingFaceEmbeddings

# LLM через Ollama (OpenAI-совместимый API)
client = AsyncOpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
llm = llm_factory("mistral", client=client)

# Локальная embedding-модель
embeddings = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8056.94it/s]


#### Шаг 4. Оценка

RAGAS оценивает качество RAG-системы через **три основные метрики**, каждая из которых — **отдельный класс**:

1. Faithfulness
2. ContextRelevance
3. AnswerRelevancy

```
Все три метрики возвращают **число от 0 до 1** (чем ближе к 1 — тем лучше).
Используются **одновременно**, чтобы оценить RAG-пайплайн целиком.
```

In [4]:
from pydantic import BaseModel
from ragas.metrics.collections import Faithfulness, AnswerRelevancy, ContextRelevance

class Result(BaseModel):
    faithfulness: float
    answer_relevancy: float
    context_relevance: float

async def evaluate_sample(sample) -> Result:
    faith = await Faithfulness(llm=llm).ascore(
        user_input=sample.user_input,
        response=sample.response,
        retrieved_contexts=sample.retrieved_contexts
    )
    ar = await AnswerRelevancy(llm=llm, embeddings=embeddings).ascore(
        user_input=sample.user_input,
        response=sample.response
    )
    cr = await ContextRelevance(llm=llm, embeddings=embeddings).ascore(
        user_input=sample.user_input,
        retrieved_contexts=sample.retrieved_contexts
    )
    return Result(
        faithfulness=faith.value,
        answer_relevancy=ar.value,
        context_relevance=cr.value
    )

async def main():
    tasks = [evaluate_sample(sample) for sample in dataset.samples]
    results = await asyncio.gather(*tasks)
    for i, res in enumerate(results):
        print(f"Sample {i+1}: {res}")

In [5]:
await main()

Sample 1: faithfulness=1.0 answer_relevancy=0.04573759761312216 context_relevance=1.0
Sample 2: faithfulness=1.0 answer_relevancy=0.4654225894087107 context_relevance=1.0


### Почему answer_relevancy может быть низким?

* **Embedding-модель** (all-MiniLM-L6-v2) слабо улавливает технические нюансы,
* **Формулировка ответа** отличается от вопроса («инструмент» ↔ «используется»),
* **RAGAS сравнивает векторы**, а не логику — даже правильный ответ может быть «далек» в embedding-пространстве.

#### Не паникуйте из-за низкого answer_relevancy.

**Сравнивайте относительные изменения:**

— «чанкинг с overlap» vs «по абзацам» → растёт context_relevance,

— «без reranking» vs «с FlashRank» → растёт faithfulness.

### Интерпретация результатов
|Метрика|Хорошее значение|Что делать, если низкое|
|:---|:---|:---|
|faithfulness|>= 0.85|Усиливайте system-промпт: «Отвечай ТОЛЬКО по контексту»|
|context_relevance|>= 0.8|Улучшайте чанкинг, используйте reranking|
|answer_relevancy|>= 0.6|Формулируйте ответ ближе к вопросу; пробуйте bge-m3 вместо all-MiniLM-L6-v2|

### Советы
* **Интегрируйте RAGAS в CI/CD:** запускайте оценку при каждом изменении пайплайна,
* **Используйте одну и ту же LLM и embedding-модель** для всех тестов — иначе сравнение бессмысленно,
* **Не используйте phi3 или gemma4 для оценки** — они дают шумные метрики. Лучше mistral, llama3:8b,
* **Сравнивайте версии, а не абсолютные цифры** — «стало лучше или хуже?» важнее, чем «0.9 или 0.95».

In [6]:
results = [{"faithfulness": 1.0,
            "context_relevance": 0.5,
            "answer_relevancy": 0.2,}, 
           {"faithfulness": 1.0,
            "context_relevance": 0.5,
            "answer_relevancy": 0.2,}]

In [7]:
import numpy as np

def aggregate_ragas_results(results: list) -> dict:
    if len(results)==0:
        return {
            "faithfulness": 0.0,
            "context_relevance": 0.0,
            "answer_relevancy": 0.0,
        }
    else:
        return {"faithfulness": np.average([result["faithfulness"] for result in results]),
                "context_relevance": np.average([result["context_relevance"] for result in results]),
                "answer_relevancy": np.average([result["answer_relevancy"] for result in results])
               }




In [8]:
aggregate_ragas_results(results)

{'faithfulness': 1.0, 'context_relevance': 0.5, 'answer_relevancy': 0.2}

In [9]:
samples = [{"user_input": "1", "response": "2", "retrieved_contexts": []}, 
           {"user_input": "3", "response": "4", "retrieved_contexts": []}]

In [10]:
def validate_ragas_dataset(samples: list) -> bool:
    """Проверяет корректность датасета для оценки RAG-систем."""
    
    # 1. samples должен быть непустым списком
    if not isinstance(samples, list) or len(samples) == 0:
        return False
    
    required_keys = {"user_input", "response", "retrieved_contexts"}
    
    for item in samples:
        # 2. Каждый элемент — словарь
        if not isinstance(item, dict):
            return False
        
        # 3. Все три ключа должны присутствовать
        if not required_keys.issubset(item.keys()):
            return False
        
        # 4. user_input и response — непустые строки
        if not isinstance(item["user_input"], str) or not item["user_input"].strip():
            return False
        if not isinstance(item["response"], str) or not item["response"].strip():
            return False
        
        # 5. retrieved_contexts — список строк (пустой допустим)
        contexts = item["retrieved_contexts"]
        if not isinstance(contexts, list):
            return False
        if not all(isinstance(ctx, str) for ctx in contexts):
            return False
    
    return True

In [11]:
validate_ragas_dataset(samples)

True